# QLoRA: Efficient Finetuning of Quantized LLMs

[QLoRA](https://arxiv.org/abs/2305.14314) is a technique primarily designed to further reduce the training memory requirement of LoRA. In QLoRA, the LoRA backbone weights are quantized to reduce the model footprint.

The main features of Quark's QLoRA:

1. QLoRA uses low-rank matrices to represent changes in model parameters, fine-tuning pre-trained language models by introducing additional low-rank parameters, thereby reducing the number of parameters that need to be updated. This method maintains model performance while significantly reducing the storage and computational resources required for training.
2. Quantizing model weights typically involves converting floating-point parameters into low-bit integers to reduce memory usage and speed up computation. During the training phase, the model's weights are quantized and locked (e.g., the weights of linear layers and corresponding quantization parameters such as scale and zero point), and only the adapter matrices are trained. This reduces the number of parameters optimized during the training phase.
3. After training, we fold the adapter weights back into the weights of the linear layer. After merging weights, the model's test metric still shows satisfactory results. As adapters are absorbed into weight, the quantized LLMs are suitable for accelerated deployment.

## Highlight Overview

We conduct `mxfp4` quantization. We simply take the `Llama-3.2-1B-Instruct` as an example to conduct the experiments. After PTQ, we added the adapter for every Linear and performed the QAT training. After training, the adapter_A and adapter_B are merged into Linear's weight. The final PPL metric results show the effectiveness of the QLoRA tools.

- **Pipeline overview :** Prepare LLM model -> PTQ -> insert adapter for liner -> QLoRA training -> merge adapter into weight.
- **Save memory:** In a single GPU with 80GB memory, we test the `Llama-3.2-1B-Instruct` (less than 20GB) and `gpt-oss-20b` (less than 70GB).
- **Efficient training:** during the whole training, only the adapters are updateable. Compared to training the whole model, QLoRA needs a small amount of training data and training loops.

- **Merge adapter weight:** After training, adapters' weight will be merged into linear's weight. After merging, the model can still keep a relative high quality metric.

## Prepare Work

As mentioned above, we first conduct Quark PTQ, and then prepare the training dataset and LLM trainer. After several steps of training, we finally merge the adapters' weight into the linear weight.

As we supply a simple example for you, with simple modification, you can directly run it, and test the metric.

- Prepare LLM & perform the corresponding PTQ: for this section, the user can refer to the Quark PTQ example.
  - Download the LLM model and save to the proper path. In this example;
  - Quark PTQ example code: Quark/examples/torch/language_modeling/llm_ptq/quantize_quark.py
  - Quark PTQ tutorials: Quark/examples/torch/language_modeling/llm_ptq/example_quark_torch_llm_ptq.rst
- Prepare the Training Dataset & the Training & training parameters.

  - Training Dataset: We use nvidia/Daring-Anteater as a fine-tune dataset. User can refer to [nvidia/Daring-Anteater](nvidia/Daring-Anteater)

## An End-to-End example of QLoRA training

### Step 1: import the necessary lib:

In [ ]:
import sys
from pathlib import Path

import transformers
from transformers import AutoProcessor, Trainer

sys.path.append("./../../")
from utils import DataArguments, PTQQuantArguments, TrainingArguments, make_supervised_data_module

from quark.contrib.llm_eval import eval_model
from quark.torch import LLMTemplate, ModelQuantizer
from quark.torch.utils.llm import get_calib_dataloader, get_model, get_tokenizer, prepare_for_moe_quant

### Step 2: Prepare args for PTQ and QLoRA training :

For better demonstration, we predefine the majority hyperparameters. And for training, we only use a small part of the dataset and a small number of training steps. For better training results, the user can adjust the learning parameters and other hyperparameters for better performance.

In [ ]:
# NOTE as runing the JuPyter Notebook will have some not desired args, so we delete.
if len(sys.argv) >= 1 and sys.argv[1].startswith("--f"):
    del sys.argv[1]

parser = transformers.HfArgumentParser((DataArguments, TrainingArguments, PTQQuantArguments))
data_args, training_args, quant_args = parser.parse_args_into_dataclasses()

After download the training dataset and LLM, assign the dataset path and LLM model path.

In [ ]:
data_args.cache_dir = "./hug_data"
quant_args.model_dir = "./pretrained_models/meta-llama/Llama-3.2-1B-Instruct/"

### Step 3: Perform the Quark PTQ

We first conduct the PTQ, we use `mxfp4` as the default quantization config, the detail config is located in `./utils.py`. As the PTQ pipeline is mainly copied from the Quark official PTQ example, for more information users can refer to: `examples/torch/language_modeling/llm_ptq/quantize_quark.py`

#### Step 3.1 Load the LLM and perform modification (if necessary).

NOTE: for some MOE models, we need to replace the expert part with Linears, so as to perform calibration.


In [ ]:
device = quant_args.device
model, model_dtype = get_model(
    quant_args.model_dir,
    quant_args.data_type,
    device,
    quant_args.multi_gpu,
    quant_args.multi_device,
    quant_args.model_attn_implementation,
    trust_remote_code=quant_args.trust_remote_code,
)

# prepare_for_moe_quant(model, quant_args.quant_algo)
prepare_for_moe_quant(model)

# model_type = get_model_type(model)
model_type = model.config.model_type if hasattr(model.config, "model_type") else model.config.architectures[0]
tokenizer = get_tokenizer(
    quant_args.model_dir,
    max_seq_len=quant_args.seq_len,
    model_type=model_type,
    trust_remote_code=quant_args.trust_remote_code,
)

multimodal = True if model_type in ["mllama", "llama4", "gemma3_mllm"] else False
if multimodal:
    processor = AutoProcessor.from_pretrained(quant_args.model_dir)
    if quant_args.model_export is not None:
        export_dir = Path(quant_args.quant_out_dir)
        export_dir.mkdir(parents=True, exist_ok=True)
        processor.save_pretrained(quant_args.quant_out_dir)

#### Step 3.2 Define PTQ calibration data loader.

In [ ]:
main_device = model.device if quant_args.multi_gpu or quant_args.multi_device else quant_args.device
calib_dataloader = get_calib_dataloader(
    dataset_name=quant_args.calib_dataset,
    processor=processor if multimodal else None,
    tokenizer=tokenizer,
    batch_size=quant_args.batch_size,
    num_calib_data=quant_args.num_calib_data,
    seqlen=quant_args.seq_len,
    device=main_device,
)

#### Step 3.3 Prepare PTQ configuration and Perform PTQ.


In [ ]:
model_config_type = model.config.model_type if hasattr(model.config, "model_type") else model.config.architectures[0]
template = LLMTemplate.get(model_config_type)
# add layer_quant_config
layer_config, algo_configs = {}, {}  # NOTE for this example, skip layer_config & algo_configs

quant_config = template.get_config(
    scheme=quant_args.quant_scheme,
    algorithm=quant_args.quant_algo,
    kv_cache_scheme=quant_args.kv_cache_dtype,
    min_kv_scale=quant_args.min_kv_scale,
    layer_config=layer_config,
    attention_scheme=quant_args.attention_dtype,
    exclude_layers=quant_args.exclude_layers,
)

quantizer = ModelQuantizer(quant_config, quant_args.multi_device)
model = quantizer.quantize_model(model, calib_dataloader)
quant_args.exclude_layers = quantizer.config.exclude

#### Step 3.4 Test the PPL after PTQ (Optional)


In [ ]:
quant_args.use_ppl_eval_model = True
eval_model(
    quant_args,
    model,
    main_device,
    save_metrics_to_csv=quant_args.save_metrics_to_csv,
    output_dir=quant_args.metrics_output_dir,
    multimodal=multimodal,
)

#### Step4.1 Modify the LLM & make only adapter trainable

We replace every `QuantLinear` with `QLoRaQuantLinear`. As `QLoRaQuantLinear` contains the adapter layer, which is designed for QLoRA training.

In [ ]:
# replace every QuantLinear to QLoRaQuantLinear
from qlora_training import trans_quant_linear_2_qlora_quantLinear

trans_quant_linear_2_qlora_quantLinear(model)

In [ ]:
# only make the adapter's param trainable
from qlora_training import disable_adapters, mark_only_lora_layer_as_trainable

mark_only_lora_layer_as_trainable(model)
# take the adapter take effect in the forward path
disable_adapters(model, False)

#### Step4.2 Prepare training dataset and Trainer 

In [ ]:
train_tokenizer = transformers.AutoTokenizer.from_pretrained(
    quant_args.model_dir, model_max_length=training_args.model_max_length
)
tokenizer.pad_token_id = tokenizer.eos_token_id

data_module = make_supervised_data_module(
    dataset=data_args.train_dataset,
    tokenizer=train_tokenizer,
    cache_dir=data_args.cache_dir,
    train_size=data_args.train_size,
    eval_size=data_args.eval_size,
)

# to save GPU memory during training.
model.enable_input_require_grads()
if model.supports_gradient_checkpointing:
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    print(f"Gradient Checkpointing: {model.is_gradient_checkpointing}")

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    **data_module,
)

In [ ]:
trainer.train()

#### Step4.3 Merge the adapter weight into linear weight

Merge the lora_B @ lora_A into linear's weight, this can let the model more compact and easy to deploy.

In [ ]:
from qlora_training import merge_weight

merge_weight(trainer.model)

In [ ]:
# freeze the model quantizer
model = quantizer.freeze(model)

In [ ]:
print("\n[INFO]: Evaluating ...")
quant_args.use_ppl_eval_model = True
eval_model(
    quant_args,
    model,
    main_device,
    save_metrics_to_csv=quant_args.save_metrics_to_csv,
    output_dir=quant_args.metrics_output_dir,
    multimodal=multimodal,
)

### Appendix:

We also provide a end-to-end example code, user can select the proper arguments to conduct the experiments.

```shell
export CUDA_VISIBLE_DEVICES=7
cd /Quark/examples/torch/language_modeling/llm_qat/qlora_training/
# using the Llama-3.2-1B-Instruct as example 

python qlora_training.py 
    --model_dir={PATH}/pretrained_models/meta-llama/Llama-3.2-1B-Instruct 
    --cache_dir={PATH}/hug_data 
    --kv_cache_dtype=fp8  
    --quant_scheme=mxfp4 
    --num_calib_data=128  
    --train_size=100 
    --eval_size=10 
    --learning_rate=1e-4  
    --lr_scheduler_type=linear
    --train_dataset=Daring-Anteater
```



### Some results

| Model              | Llama-3.2-1B-Instruct | gpt-oss-20b/ |
| ------------------ | --------------------- | ------------ |
| ORG Model PPL      | 13.4199               | null         |
| PTQ‘s PPL          | 18.8498               | null         |
| QLoRA's PPL        | 16.9987               | null         |
| PTQ's GPU memory   | 3620MB                | 48GB         |
| QLoRA's GPU memory | 18GB                  | 66GB         |

### NOTE： 

   In the future, we will conduct more experiments. Next Step, we will focus on the `gpt-oss-20b` and prepare the corresponding test metrics.